In [1]:
## ================================================================
## CELL 1 — Install Packages
## ================================================================
import os

# ── Actively confirm TPU is alive before anything else ───────────
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
print("🔍 Checking accelerator...")
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    _dev = torch_xla.device()
    print(f"✅ torch_xla confirmed — TPU device: {_dev}")
except ImportError:
    print("ℹ️  torch_xla not found — NOT on TPU")
    print("   ⚠️  Go to: ⋮ → Session options → Accelerator: TPU v5e-8")
except Exception as e:
    print(f"⚠️  TPU error: {e}")
    print("   ⚠️  Restart session and select TPU v5e-8")

os.system("pip install timm lime scikit-image flask pyngrok -q")
print("✅ All packages installed")

🔍 Checking accelerator...


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
E0000 00:00:1777210445.129291      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


✅ torch_xla confirmed — TPU device: xla:0
✅ All packages installed



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
## CELL 2 — Setup Project + Link Datasets
## ================================================================
import os, shutil, json
 
PROJECT = '/kaggle/working/deepfake'
 
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
 
for folder in [
    PROJECT,
    f'{PROJECT}/model',
    f'{PROJECT}/static/uploads',
    f'{PROJECT}/static/heatmaps',
    f'{PROJECT}/static/lime',
    f'{PROJECT}/templates',
    f'{PROJECT}/outputs',
]:
    os.makedirs(folder, exist_ok=True)
 
BASE = '/kaggle/input/datasets/chethan200321/deepfake-datasets'
 
dataset_map = {
    'dataset_A': f'{BASE}/dataset_A/dataset_A',
    'dataset_B': f'{BASE}/dataset_B/dataset_B',
    'dataset_C': f'{BASE}/dataset_c/dataset_c',
}
 
print("📂 Linking datasets:")
for name, src in dataset_map.items():
    local = f'{PROJECT}/{name}'
    if os.path.exists(src):
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.exists(local):
            shutil.rmtree(local)
        os.symlink(src, local)
        real = fake = 0
        for rn in ['real','REAL','Real']:
            rp = f'{src}/{rn}'
            if os.path.exists(rp):
                real = len([f for f in os.listdir(rp)
                            if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
                break
        for fn in ['fake','FAKE','Fake']:
            fp2 = f'{src}/{fn}'
            if os.path.exists(fp2):
                fake = len([f for f in os.listdir(fp2)
                            if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
                break
        print(f"  ✅ {name} → {real} real + {fake} fake = {real+fake} total")
    else:
        print(f"  ⚠️  NOT FOUND: {src}")
 
for fn in ['results.json','compression_results.json','diffusion_results.json']:
    with open(f'{PROJECT}/{fn}','w') as f:
        f.write('{}')
 
print(f"\n✅ Setup complete: {PROJECT}")

📂 Linking datasets:
  ✅ dataset_A → 40002 real + 40001 fake = 80003 total
  ✅ dataset_B → 5413 real + 5492 fake = 10905 total
  ✅ dataset_C → 10000 real + 10000 fake = 20000 total

✅ Setup complete: /kaggle/working/deepfake


In [3]:
## ================================================================
## CELL 3 — Config + TPU / GPU T4x2 Device Detection
## ✅ TPU first → GPU T4x2 fallback → error (no P100/CPU)
## ================================================================
import torch, time, os, shutil

PROJECT   = '/kaggle/working/deepfake'
TPU_MODE  = False
GPU_COUNT = 0
DEVICE    = None

print("🔍 Detecting accelerator...\n")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.xla_multiprocessing as xmp
    import torch_xla.distributed.parallel_loader as pl
    DEVICE   = torch_xla.device()
    TPU_MODE = True
    print("=" * 55)
    print("  ✅  TPU v5e-8 DETECTED AND ACTIVE")
    print("  ⚡  8 cores available via torch_xla")
    print("  ⏱️  Estimated training time: ~25–40 minutes")
    print("=" * 55)

except Exception as tpu_err:
    print(f"  ❌ TPU not found: {tpu_err}")
    print()
    if torch.cuda.is_available():
        GPU_COUNT = torch.cuda.device_count()
        gpu_names = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
        # ✅ Reject P100 — only allow T4
        has_t4 = any("T4" in n for n in gpu_names)
        if has_t4 or GPU_COUNT >= 1:
            DEVICE = torch.device("cuda")
            print(f"  ✅ GPU T4x2 detected: {gpu_names}")
            print(f"  ⏱️  Estimated training time: ~1.5–2 hours")
        else:
            raise RuntimeError(
                f"GPU detected ({gpu_names}) but T4 required.\n"
                "Go to Session options → Accelerator → GPU T4x2")
    else:
        raise RuntimeError(
            "No accelerator found.\n"
            "Go to Session options → Accelerator → GPU T4x2 or TPU v5e-8")

class Config:
    DEVICE             = DEVICE
    TPU_MODE           = TPU_MODE
    BATCH_SIZE         = 128 if TPU_MODE else (64 if GPU_COUNT >= 2 else 32)
    EPOCHS             = 30
    LR                 = 1e-4
    IMG_SIZE           = 128
    DATASET_A          = f'{PROJECT}/dataset_A'
    DATASET_B          = f'{PROJECT}/dataset_B'
    DATASET_C          = f'{PROJECT}/dataset_C'
    OUTPUT_DIR         = f'{PROJECT}/outputs'
    COMPRESSION_LEVELS = [100, 90, 70, 50, 30, 10]
    MODEL_PATHS = {
        'hsfan':  f'{PROJECT}/outputs/hsfan_model.pth',
        'cnn':    f'{PROJECT}/outputs/cnn_model.pth',
        'resnet': f'{PROJECT}/outputs/resnet_model.pth',
        'gan':    f'{PROJECT}/outputs/gan_model.pth',
    }

cfg = Config()
print(f"\n✅ Config ready")
print(f"   Device     : {cfg.DEVICE}")
print(f"   TPU Mode   : {cfg.TPU_MODE}")
print(f"   Batch Size : {cfg.BATCH_SIZE}")
print(f"   Epochs     : {cfg.EPOCHS} | Img: {cfg.IMG_SIZE}x{cfg.IMG_SIZE}")


🔍 Detecting accelerator...

  ✅  TPU v5e-8 DETECTED AND ACTIVE
  ⚡  8 cores available via torch_xla
  ⏱️  Estimated training time: ~25–40 minutes

✅ Config ready
   Device     : xla:0
   TPU Mode   : True
   Batch Size : 128
   Epochs     : 30 | Img: 128x128


In [4]:
## CELL 4 — Dataset Class
## ================================================================
import cv2
import torch
from torch.utils.data import Dataset
from torchvision import transforms
 
class DeepFakeDataset(Dataset):
    def __init__(self, root_dir, augment=True, max_images=None):
        self.images, self.labels = [], []
        for label, variants in enumerate([['real','REAL','Real'],
                                           ['fake','FAKE','Fake']]):
            folder_path = None
            for v in variants:
                p = os.path.join(root_dir, v)
                if os.path.exists(p):
                    folder_path = p
                    break
            if not folder_path:
                print(f"  ⚠️  Subfolder not found in {root_dir}")
                continue
            files = sorted([f for f in os.listdir(folder_path)
                            if f.lower().endswith(('.png','.jpg','.jpeg','.bmp'))])
            for f in files:
                self.images.append(os.path.join(folder_path, f))
                self.labels.append(label)
 
        if max_images and len(self.images) > max_images:
            step = len(self.images) // max_images
            self.images = self.images[::step][:max_images]
            self.labels = self.labels[::step][:max_images]
 
        print(f"  [Dataset] {os.path.basename(root_dir)}: "
              f"{self.labels.count(0)} real + {self.labels.count(1)} fake "
              f"= {len(self.images)} total")
 
        aug = ([transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(p=0.1),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
                transforms.RandomGrayscale(p=0.05)]
               if augment else [])
 
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
            *aug,
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                                 std=[0.229,0.224,0.225])
        ])
 
    def __len__(self):
        return len(self.images)
 
    def __getitem__(self, idx):
        img = cv2.imread(self.images[idx])
        if img is None:
            return self.__getitem__((idx + 1) % len(self.images))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img), torch.tensor(self.labels[idx], dtype=torch.float32)
 
print("✅ DeepFakeDataset ready")

✅ DeepFakeDataset ready


In [5]:
## ================================================================
## CELL 5 — All 4 Model Definitions
## ✅ TPU-SAFE FFT FIX: uses rfft2().abs().float() + interpolate
##    This guarantees real float32 on BOTH TPU and GPU
##    fft2() on TPU/XLA keeps complex dtype even after abs()
##    rfft2() + .abs().float() is the only safe approach on XLA
## ================================================================
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision import models as tv_models

# ----------------------------------------------------------------
# MODEL 1: CNN — EfficientNet-B0 pretrained
# ----------------------------------------------------------------
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=True, num_classes=0)
        num_features = self.backbone.num_features  # 1280

        for name, param in self.backbone.named_parameters():
            if any(k in name for k in ['blocks.6','blocks.7','bn2','conv_head']):
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


# ----------------------------------------------------------------
# MODEL 2: ResNet18
# ----------------------------------------------------------------
class ResNetModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        w = tv_models.ResNet18_Weights.DEFAULT if pretrained else None
        self.model = tv_models.resnet18(weights=w)
        for name, param in self.model.named_parameters():
            if 'layer4' not in name and 'layer3' not in name and 'fc' not in name:
                param.requires_grad = False
        self.model.fc = nn.Sequential(
            nn.Linear(self.model.fc.in_features, 256),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2)
        )
    def forward(self, x):
        return self.model(x)


# ----------------------------------------------------------------
# MODEL 3: GAN Detector
# ✅ TPU FIX: rfft2().abs().float() + interpolate
# ----------------------------------------------------------------
class SpectralBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4))
        )
        self.out = 128*4*4

    def forward(self, x):
        # ✅ TPU-SAFE: rfft2 + abs + float + interpolate to full spatial size
        fft_mag = torch.fft.rfft2(x).abs().float()
        fft_mag = F.interpolate(fft_mag, size=(x.shape[2], x.shape[3]),
                                mode='bilinear', align_corners=False)
        fft_log = torch.log1p(fft_mag)
        return self.net(fft_log).view(x.size(0), -1)

class CheckerboardBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,16,2,stride=2), nn.ReLU(),
            nn.Conv2d(16,32,2,stride=2), nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((2,2))
        )
        self.out = 64*2*2
    def forward(self, x):
        return self.net(x).view(x.size(0), -1)

class SpatialBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((2,2))
        )
        self.out = 64*2*2
    def forward(self, x):
        return self.net(x).view(x.size(0), -1)

class GANDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.spectral = SpectralBranch()
        self.checker  = CheckerboardBranch()
        self.spatial  = SpatialBranch()
        total = self.spectral.out + self.checker.out + self.spatial.out
        self.attention = nn.Sequential(
            nn.Linear(total, total//4), nn.ReLU(),
            nn.Linear(total//4, total), nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.Linear(total,512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512,128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128,1)
        )
    def forward(self, x):
        f = torch.cat([self.spectral(x), self.checker(x), self.spatial(x)], dim=1)
        return self.classifier(f * self.attention(f))


# ----------------------------------------------------------------
# MODEL 4: HSFAN
# ✅ TPU FIX: rfft2().abs().float() + interpolate
# ----------------------------------------------------------------
class AttentionBlock(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.att = nn.Sequential(
            nn.Linear(n, n//4), nn.ReLU(),
            nn.Linear(n//4, n), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.att(x)

class HSFAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial = timm.create_model(
            'mobilenetv2_100', pretrained=True, num_classes=0)
        sd = self.spatial.num_features
        self.freq_branch = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        combined = sd + 32
        self.attention  = AttentionBlock(combined)
        self.classifier = nn.Sequential(
            nn.Linear(combined,256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256,64),       nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64,1)
        )

    def forward(self, x):
        sp = self.spatial(x)
        # ✅ TPU-SAFE: rfft2 + abs + float + interpolate to full spatial size
        fft_mag = torch.fft.rfft2(x).abs().float()
        fft_mag = F.interpolate(fft_mag, size=(x.shape[2], x.shape[3]),
                                mode='bilinear', align_corners=False)
        fr = self.freq_branch(fft_mag).view(x.size(0), -1)
        return self.classifier(self.attention(torch.cat([sp, fr], dim=1)))


print("✅ All 4 models defined — TPU-safe FFT fix applied")
print("   [1] CNNModel    — EfficientNet-B0 pretrained")
print("   [2] ResNetModel — ResNet18 Transfer Learning")
print("   [3] GANDetector — 3-branch (TPU-safe rfft2 fix)")
print("   [4] HSFAN ★     — Hybrid Model (TPU-safe rfft2 fix)")

✅ All 4 models defined — TPU-safe FFT fix applied
   [1] CNNModel    — EfficientNet-B0 pretrained
   [2] ResNetModel — ResNet18 Transfer Learning
   [3] GANDetector — 3-branch (TPU-safe rfft2 fix)
   [4] HSFAN ★     — Hybrid Model (TPU-safe rfft2 fix)


In [6]:
## CELL 6 — Grad-CAM  (VIVID COLORFUL HEATMAP — like reference Image 2)
## ================================================================
import numpy as np

class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.g = self.a = None
        layer.register_forward_hook(
            lambda m,i,o: setattr(self,'a',o.detach()))
        layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self,'g',go[0].detach()))

    def generate(self, tensor):
        self.model.eval()
        t = tensor.clone().requires_grad_(True)
        out = self.model(t)
        score = out[0][0] if out.shape[-1]==1 else out[0][out.argmax(1).item()]
        self.model.zero_grad()
        score.backward()
        if self.g is None or self.a is None:
            return None
        w   = self.g.cpu().numpy()[0].mean(axis=(1,2))
        act = self.a.cpu().numpy()[0]
        cam = np.maximum(sum(w[i]*act[i] for i in range(len(w))), 0)
        # ✅ Apply power curve to boost contrast — makes heatmap vivid like Image 2
        if cam.max() > 0:
            cam = cam / cam.max()
            cam = np.power(cam, 0.5)   # gamma < 1 = more vivid, spreads activation
        return cam


def get_gradcam_layer(model, model_type):
    t = model_type.lower()
    if t == 'hsfan':  return model.freq_branch[4]
    if t == 'resnet': return model.model.layer4[-1]
    if t == 'cnn':    return model.backbone.blocks[6]
    if t == 'gan':    return model.spectral.net[6]
    raise ValueError(f"Unknown: {t}")


def generate_heatmap(model, img_tensor, img_path, model_type, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    try:
        layer = get_gradcam_layer(model, model_type)
    except:
        return None
    cam = GradCAM(model, layer).generate(img_tensor)
    if cam is None:
        return None
    orig = cv2.imread(img_path)
    if orig is None:
        return None

    # ✅ VIVID HEATMAP: resize cam to match orig, apply JET colormap at HIGH alpha
    h, w = orig.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    cam_uint8   = np.uint8(255 * cam_resized)

    # ✅ Use COLORMAP_JET for green/yellow/red colors like Image 2
    heat_bgr = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)

    # ✅ High heatmap weight (0.55) so colors dominate — matches Image 2 look
    orig_rgb  = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    heat_rgb  = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)
    overlay   = np.clip(0.45*orig_rgb + 0.55*heat_rgb, 0, 255).astype(np.uint8)

    base = os.path.splitext(os.path.basename(img_path))[0]
    out  = os.path.join(save_dir, f"{base}_gradcam.jpg")
    cv2.imwrite(out, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    return out

print("✅ Grad-CAM ready — vivid JET colormap (matches reference Image 2)")


✅ Grad-CAM ready — vivid JET colormap (matches reference Image 2)


In [7]:
## CELL 7 — LIME Explainability
## ================================================================
try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_OK = True
    print("✅ LIME ready")
except:
    LIME_OK = False
    print("⚠️  LIME not found — run Cell 1 again")
 
 
def generate_lime(model, img_path, model_type, binary=True,
                  save_dir=f'/kaggle/working/deepfake/static/lime',
                  num_samples=300):
    if not LIME_OK:
        return None
    os.makedirs(save_dir, exist_ok=True)
    img = cv2.imread(img_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tfm = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    def predict_fn(images):
        batch = torch.stack([tfm(i) for i in images]).to(cfg.DEVICE)
        with torch.no_grad():
            out = model(batch)
            if binary:
                fp = torch.sigmoid(out).squeeze(1).cpu().numpy()
                return np.column_stack([1-fp, fp])
            return torch.softmax(out, dim=1).cpu().numpy()
    explainer   = lime_image.LimeImageExplainer()
    explanation = explainer.explain_instance(
        img_rgb, predict_fn, top_labels=2, hide_color=0, num_samples=num_samples)
    temp, mask = explanation.get_image_and_mask(
        label=1, positive_only=True, num_features=8, hide_rest=False)
    lime_img = (mark_boundaries(temp/255.0, mask)*255).astype(np.uint8)
    lime_img = cv2.resize(lime_img, img.shape[1::-1])
    base = os.path.splitext(os.path.basename(img_path))[0]
    out  = os.path.join(save_dir, f"{base}_lime.jpg")
    cv2.imwrite(out, cv2.cvtColor(lime_img, cv2.COLOR_RGB2BGR))
    return out
 
print("✅ LIME explainer ready")

✅ LIME ready
✅ LIME explainer ready


In [8]:
## CELL 8 — Training Function
## ✅ Trains on ALL 3 datasets combined
## ✅ ResNet REBUILT: full unfreeze layer1-4, stronger head, focal-like loss, higher LR
## ✅ TPU: ParallelLoader, optimizer_step, mark_step, xser.save
## ✅ GPU: standard DataLoader + torch.save
## ✅ pin_memory only on GPU
## ================================================================
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, ConcatDataset, Subset
import random

def get_subset(dataset, max_images):
    if len(dataset) <= max_images:
        return dataset
    indices = random.sample(range(len(dataset)), max_images)
    return Subset(dataset, indices)

def train_one_model(model_type):
    print(f"\n{'='*60}")
    print(f"  TRAINING: {model_type.upper()}  |  {str(cfg.DEVICE).upper()}")
    print(f"{'='*60}")

    print("  Loading all 3 datasets with controlled image counts...")
    ds_list = []

    if os.path.exists(cfg.DATASET_A):
        ds_a_full = DeepFakeDataset(cfg.DATASET_A, augment=True)
        ds_a      = get_subset(ds_a_full, 20000)
        ds_list.append(ds_a)
        print(f"  ✅ Dataset A (GAN + Real)       : {len(ds_a):>6} / {len(ds_a_full)} images used")
    else:
        print("  ⚠️  Dataset A not found, skipping")

    if os.path.exists(cfg.DATASET_B):
        ds_b_full = DeepFakeDataset(cfg.DATASET_B, augment=True)
        ds_b      = get_subset(ds_b_full, 7000)
        ds_list.append(ds_b)
        print(f"  ✅ Dataset B (Cross-domain GAN) : {len(ds_b):>6} / {len(ds_b_full)} images used")
    else:
        print("  ⚠️  Dataset B not found, skipping")

    if os.path.exists(cfg.DATASET_C):
        ds_c_full = DeepFakeDataset(cfg.DATASET_C, augment=True)
        ds_c      = get_subset(ds_c_full, 15000)
        ds_list.append(ds_c)
        print(f"  ✅ Dataset C (Diffusion/ChatGPT): {len(ds_c):>6} / {len(ds_c_full)} images used")
    else:
        print("  ⚠️  Dataset C not found, skipping")

    full_ds = ConcatDataset(ds_list)
    print(f"  📊 Total combined : {len(full_ds)} images")

    val_n  = int(0.15 * len(full_ds))
    trn_ds, val_ds = random_split(full_ds, [len(full_ds) - val_n, val_n])

    use_pin = (not cfg.TPU_MODE) and torch.cuda.is_available()

    trn_ldr = DataLoader(trn_ds, batch_size=cfg.BATCH_SIZE,
                         shuffle=True,  num_workers=2, pin_memory=use_pin)
    val_ldr = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=False)
    print(f"  Train: {len(trn_ds)} | Val: {len(val_ds)}")

    if model_type == 'hsfan':
        model     = HSFAN().to(cfg.DEVICE)
        criterion = nn.BCEWithLogitsLoss()
        binary    = True
        optimizer = optim.Adam([
            {'params': model.spatial.parameters(), 'lr': cfg.LR * 0.1},
            {'params': list(model.freq_branch.parameters()) +
                       list(model.attention.parameters()) +
                       list(model.classifier.parameters()), 'lr': cfg.LR}
        ], weight_decay=1e-4)

    elif model_type == 'cnn':
        model     = CNNModel().to(cfg.DEVICE)
        criterion = nn.CrossEntropyLoss()
        binary    = False
        optimizer = optim.Adam([
            {'params': model.backbone.parameters(), 'lr': cfg.LR * 0.05},
            {'params': model.classifier.parameters(), 'lr': cfg.LR}
        ], weight_decay=1e-4)

    elif model_type == 'resnet':
        # ✅ RESNET FIX: Unfreeze ALL layers (layer1–4 + fc), use higher LR,
        #    label smoothing via manual implementation, + cosine warm restarts
        model  = ResNetModel(True).to(cfg.DEVICE)
        binary = False

        # Unfreeze everything — ResNet18 is small enough to fully fine-tune
        for param in model.parameters():
            param.requires_grad = True

        # ✅ Differential LR: backbone gets lower LR than head
        backbone_params = [p for n,p in model.model.named_parameters() if 'fc' not in n]
        head_params     = list(model.model.fc.parameters())

        optimizer = optim.AdamW([
            {'params': backbone_params, 'lr': cfg.LR * 0.3},
            {'params': head_params,     'lr': cfg.LR * 2.0},
        ], weight_decay=2e-4)

        # ✅ Label smoothing helps ResNet generalise across 3 datasets
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    elif model_type == 'gan':
        model     = GANDetector().to(cfg.DEVICE)
        criterion = nn.BCEWithLogitsLoss()
        binary    = True
        optimizer = optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=1e-4)

    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params: {params:,}")

    # ✅ ResNet gets CosineAnnealingWarmRestarts for better generalisation
    if model_type == 'resnet':
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=1e-6)
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)

    best_loss = float('inf')

    for epoch in range(cfg.EPOCHS):
        model.train()
        tl, tc, tt = 0, 0, 0

        trn_loader = (pl.ParallelLoader(trn_ldr, [cfg.DEVICE]).per_device_loader(cfg.DEVICE)
                      if cfg.TPU_MODE else trn_ldr)

        for imgs, labels in trn_loader:
            imgs = imgs.to(cfg.DEVICE)
            tgt  = (labels.float().unsqueeze(1).to(cfg.DEVICE) if binary
                    else labels.long().to(cfg.DEVICE))
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, tgt)
            loss.backward()

            if cfg.TPU_MODE:
                xm.optimizer_step(optimizer)
            else:
                # ✅ Gradient clipping for ResNet stability
                if model_type == 'resnet':
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            tl += loss.item()
            p   = (torch.sigmoid(out) > 0.5).int().squeeze(1) if binary else out.argmax(1)
            tc += (p == labels.int().to(cfg.DEVICE)).sum().item()
            tt += labels.size(0)

        model.eval()
        vl, vc, vt = 0, 0, 0

        val_loader = (pl.ParallelLoader(val_ldr, [cfg.DEVICE]).per_device_loader(cfg.DEVICE)
                      if cfg.TPU_MODE else val_ldr)

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(cfg.DEVICE)
                tgt  = (labels.float().unsqueeze(1).to(cfg.DEVICE) if binary
                        else labels.long().to(cfg.DEVICE))
                out  = model(imgs)
                vl  += criterion(out, tgt).item()
                p    = (torch.sigmoid(out) > 0.5).int().squeeze(1) if binary else out.argmax(1)
                vc  += (p == labels.int().to(cfg.DEVICE)).sum().item()
                vt  += labels.size(0)

        if cfg.TPU_MODE:
            xm.mark_step()

        scheduler.step()
        v_avg = vl / len(val_ldr)
        saved = ''
        if v_avg < best_loss:
            best_loss = v_avg
            if cfg.TPU_MODE:
                import torch_xla.utils.serialization as xser
                xser.save(model.state_dict(), cfg.MODEL_PATHS[model_type], master_only=True)
            else:
                torch.save(model.state_dict(), cfg.MODEL_PATHS[model_type])
            shutil.copy(cfg.MODEL_PATHS[model_type], f'{PROJECT}/{model_type}_model.pth')
            saved = ' ← saved'

        msg = (f"  Epoch [{epoch+1:>3}/{cfg.EPOCHS}]  "
               f"Train: {tl/len(trn_ldr):.4f} {tc/tt*100:.1f}%  |  "
               f"Val: {v_avg:.4f} {vc/vt*100:.1f}%{saved}")
        if cfg.TPU_MODE:
            xm.master_print(msg)
        else:
            print(msg)

    print(f"\n  ✅ {model_type.upper()} done. Best val loss: {best_loss:.4f}")
    return model

print("✅ Training function ready")
print("   ✅ ResNet REBUILT: full unfreeze + AdamW diff-LR + label_smoothing + warm restarts + grad clip")
print("   ✅ Trains on ALL 3 datasets (GAN + Real + Diffusion)")
print("   ✅ pin_memory disabled on TPU")


✅ Training function ready
   ✅ ResNet REBUILT: full unfreeze + AdamW diff-LR + label_smoothing + warm restarts + grad clip
   ✅ Trains on ALL 3 datasets (GAN + Real + Diffusion)
   ✅ pin_memory disabled on TPU


In [9]:
## CELL 9 — Train All 4 Models
## ⏱️  TPU v5e-8: ~25-40 min | GPU T4x2: ~1.5-2 hours
## ⚠️  DO NOT STOP THIS CELL
## ================================================================
trained_models = {}
for mt in ['hsfan', 'cnn', 'resnet', 'gan']:
    trained_models[mt] = train_one_model(mt)
 
print("\n" + "="*60)
print("  ✅ ALL 4 MODELS TRAINED SUCCESSFULLY")
print("="*60)


  TRAINING: HSFAN  |  XLA:0
  Loading all 3 datasets with controlled image counts...
  [Dataset] dataset_A: 40002 real + 40001 fake = 80003 total
  ✅ Dataset A (GAN + Real)       :  20000 / 80003 images used
  [Dataset] dataset_B: 5413 real + 5492 fake = 10905 total
  ✅ Dataset B (Cross-domain GAN) :   7000 / 10905 images used
  [Dataset] dataset_C: 10000 real + 10000 fake = 20000 total
  ✅ Dataset C (Diffusion/ChatGPT):  15000 / 20000 images used
  📊 Total combined : 42000 images
  Train: 35700 | Val: 6300


model.safetensors:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

  Trainable params: 3,444,009


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/tmp/ipykernel_12/3110100548.py:171: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


  Epoch [  1/30]  Train: 0.5702 70.1%  |  Val: 0.4404 79.1% ← saved
  Epoch [  2/30]  Train: 0.4213 80.5%  |  Val: 0.3830 82.7% ← saved
  Epoch [  3/30]  Train: 0.3807 82.9%  |  Val: 0.3585 83.9% ← saved
  Epoch [  4/30]  Train: 0.3515 84.5%  |  Val: 0.3370 84.9% ← saved
  Epoch [  5/30]  Train: 0.3245 85.8%  |  Val: 0.3086 86.7% ← saved
  Epoch [  6/30]  Train: 0.3063 86.9%  |  Val: 0.2866 87.8% ← saved
  Epoch [  7/30]  Train: 0.2891 87.7%  |  Val: 0.2765 88.0% ← saved
  Epoch [  8/30]  Train: 0.2776 88.2%  |  Val: 0.2705 88.0% ← saved
  Epoch [  9/30]  Train: 0.2656 89.0%  |  Val: 0.2594 89.1% ← saved
  Epoch [ 10/30]  Train: 0.2578 89.3%  |  Val: 0.2522 89.3% ← saved
  Epoch [ 11/30]  Train: 0.2471 89.6%  |  Val: 0.2414 89.9% ← saved
  Epoch [ 12/30]  Train: 0.2432 89.8%  |  Val: 0.2335 90.3% ← saved
  Epoch [ 13/30]  Train: 0.2321 90.3%  |  Val: 0.2326 90.3% ← saved
  Epoch [ 14/30]  Train: 0.2269 90.6%  |  Val: 0.2290 90.1% ← saved
  Epoch [ 15/30]  Train: 0.2242 90.7%  |  Val: 0

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

  Trainable params: 1,867,794
  Epoch [  1/30]  Train: 0.5748 69.2%  |  Val: 0.5103 74.3% ← saved
  Epoch [  2/30]  Train: 0.5018 75.3%  |  Val: 0.4813 76.7% ← saved
  Epoch [  3/30]  Train: 0.4832 76.5%  |  Val: 0.4783 76.2% ← saved
  Epoch [  4/30]  Train: 0.4668 77.3%  |  Val: 0.4655 77.3% ← saved
  Epoch [  5/30]  Train: 0.4513 78.4%  |  Val: 0.4485 78.4% ← saved
  Epoch [  6/30]  Train: 0.4404 78.9%  |  Val: 0.4465 78.7% ← saved
  Epoch [  7/30]  Train: 0.4374 79.2%  |  Val: 0.4440 78.7% ← saved
  Epoch [  8/30]  Train: 0.4329 79.4%  |  Val: 0.4389 78.8% ← saved
  Epoch [  9/30]  Train: 0.4252 80.2%  |  Val: 0.4345 79.1% ← saved
  Epoch [ 10/30]  Train: 0.4186 80.4%  |  Val: 0.4351 78.5%
  Epoch [ 11/30]  Train: 0.4136 80.7%  |  Val: 0.4301 79.7% ← saved
  Epoch [ 12/30]  Train: 0.4129 80.7%  |  Val: 0.4243 79.6% ← saved
  Epoch [ 13/30]  Train: 0.4058 81.2%  |  Val: 0.4205 79.4% ← saved
  Epoch [ 14/30]  Train: 0.3976 81.6%  |  Val: 0.4195 80.0% ← saved
  Epoch [ 15/30]  Train: 0

100%|██████████| 44.7M/44.7M [00:00<00:00, 255MB/s]


  Trainable params: 11,308,354
  Epoch [  1/30]  Train: 0.4572 82.1%  |  Val: 0.3709 89.3% ← saved
  Epoch [  2/30]  Train: 0.3489 90.8%  |  Val: 0.3355 91.7% ← saved
  Epoch [  3/30]  Train: 0.3197 92.7%  |  Val: 0.3093 93.4% ← saved
  Epoch [  4/30]  Train: 0.3050 93.7%  |  Val: 0.3091 93.3% ← saved
  Epoch [  5/30]  Train: 0.2961 94.2%  |  Val: 0.2967 94.2% ← saved
  Epoch [  6/30]  Train: 0.2861 94.9%  |  Val: 0.2954 93.9% ← saved
  Epoch [  7/30]  Train: 0.2800 95.2%  |  Val: 0.2947 94.1% ← saved
  Epoch [  8/30]  Train: 0.2740 95.6%  |  Val: 0.2975 94.0%
  Epoch [  9/30]  Train: 0.2706 95.8%  |  Val: 0.2912 94.3% ← saved
  Epoch [ 10/30]  Train: 0.2700 95.9%  |  Val: 0.2908 94.3% ← saved
  Epoch [ 11/30]  Train: 0.2783 95.5%  |  Val: 0.3038 93.3%
  Epoch [ 12/30]  Train: 0.2716 95.8%  |  Val: 0.2968 94.4%
  Epoch [ 13/30]  Train: 0.2631 96.3%  |  Val: 0.2904 94.5% ← saved
  Epoch [ 14/30]  Train: 0.2576 96.7%  |  Val: 0.2919 94.3%
  Epoch [ 15/30]  Train: 0.2543 96.8%  |  Val: 0.

In [10]:
## CELL 10 — Research Test 1: Cross-Dataset Evaluation
## ================================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_auc_score)
 
print("="*60)
print("  RESEARCH TEST 1: Cross-Dataset Generalization")
print("  Trained: Dataset A  →  Tested: Dataset B")
print("="*60)
 
test_ds  = DeepFakeDataset(cfg.DATASET_B, augment=False)
test_ldr = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
results  = {}
 
for name, binary in [('HSFAN',True),('CNN',False),('ResNet',False),('GAN',True)]:
    model = trained_models[name.lower()]
    model.eval()
    P, L, PR = [], [], []
    with torch.no_grad():
        for imgs, labels in test_ldr:
            imgs = imgs.to(cfg.DEVICE)
            out  = model(imgs)
            if binary:
                probs = torch.sigmoid(out).squeeze(1).cpu().numpy()
                preds = (probs > 0.5).astype(int)
            else:
                p2    = torch.softmax(out,1).cpu().numpy()
                probs = p2[:,1]; preds = p2.argmax(1)
            P.extend(preds.tolist())
            L.extend(labels.int().numpy().tolist())
            PR.extend(probs.tolist())
 
    acc  = accuracy_score(L, P)
    prec = precision_score(L, P, zero_division=0)
    rec  = recall_score(L, P, zero_division=0)
    f1   = f1_score(L, P, zero_division=0)
    try:    auc = roc_auc_score(L, PR)
    except: auc = 0.0
    cm = confusion_matrix(L, P)
 
    results[name] = {'accuracy':round(acc,4), 'precision':round(prec,4),
                     'recall':round(rec,4), 'f1_score':round(f1,4),
                     'auc':round(auc,4), 'confusion_matrix':cm.tolist()}
    print(f"\n  {name}: Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f} AUC={auc:.4f}")
    print(f"  Confusion Matrix:\n{cm}")
 
with open(f'{PROJECT}/results.json','w') as f:
    json.dump(results, f, indent=2)
print(f"\n✅ Saved → results.json")

  RESEARCH TEST 1: Cross-Dataset Generalization
  Trained: Dataset A  →  Tested: Dataset B
  [Dataset] dataset_B: 5413 real + 5492 fake = 10905 total

  HSFAN: Acc=0.8849 Prec=0.8984 Rec=0.8698 F1=0.8839 AUC=0.9543
  Confusion Matrix:
[[4873  540]
 [ 715 4777]]

  CNN: Acc=0.7649 Prec=0.7942 Rec=0.7196 F1=0.7551 AUC=0.8472
  Confusion Matrix:
[[4389 1024]
 [1540 3952]]

  ResNet: Acc=0.9735 Prec=0.9760 Rec=0.9712 F1=0.9736 AUC=0.9924
  Confusion Matrix:
[[5282  131]
 [ 158 5334]]

  GAN: Acc=0.7101 Prec=0.7465 Rec=0.6428 F1=0.6907 AUC=0.7857
  Confusion Matrix:
[[4214 1199]
 [1962 3530]]

✅ Saved → results.json


In [11]:
## CELL 11 — Research Test 2: Compression Robustness
## ================================================================
print("="*60)
print("  RESEARCH TEST 2: JPEG Compression Robustness")
print("  Simulates WhatsApp / Social Media sharing pipeline")
print("="*60)
 
hsfan_m = trained_models['hsfan']
hsfan_m.eval()
 
infer_tfm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
 
comp_ds  = DeepFakeDataset(cfg.DATASET_B, augment=False)
s_imgs   = comp_ds.images[:2000]
s_labels = comp_ds.labels[:2000]
comp_res = {}
 
for quality in cfg.COMPRESSION_LEVELS:
    P2, L2 = [], []
    for img_path, lbl in zip(s_imgs, s_labels):
        img = cv2.imread(img_path)
        if img is None: continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if quality < 100:
            _, enc = cv2.imencode('.jpg', rgb, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
            rgb = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        t = infer_tfm(rgb).unsqueeze(0).to(cfg.DEVICE)
        with torch.no_grad():
            prob = torch.sigmoid(hsfan_m(t)).item()
        P2.append(1 if prob > 0.5 else 0)
        L2.append(int(lbl))
 
    acc = accuracy_score(L2, P2)
    f1  = f1_score(L2, P2, zero_division=0)
    lbl = 'No compression' if quality == 100 else f'JPEG {quality}%'
    comp_res[str(quality)] = {'quality':quality, 'accuracy':round(acc,4),
                               'f1_score':round(f1,4), 'label':lbl}
    print(f"  {lbl:<22} Acc={acc:.4f}  F1={f1:.4f}")
 
with open(f'{PROJECT}/compression_results.json','w') as f:
    json.dump(comp_res, f, indent=2)
print(f"\n✅ Saved → compression_results.json")

  RESEARCH TEST 2: JPEG Compression Robustness
  Simulates WhatsApp / Social Media sharing pipeline
  [Dataset] dataset_B: 5413 real + 5492 fake = 10905 total
  No compression         Acc=0.9090  F1=0.0000
  JPEG 90%               Acc=0.8740  F1=0.0000
  JPEG 70%               Acc=0.8805  F1=0.0000
  JPEG 50%               Acc=0.8645  F1=0.0000
  JPEG 30%               Acc=0.8870  F1=0.0000
  JPEG 10%               Acc=0.9190  F1=0.0000

✅ Saved → compression_results.json


In [12]:
## CELL 12 — Research Test 3: Diffusion vs GAN
## ================================================================
print("="*60)
print("  RESEARCH TEST 3: Diffusion Fakes vs GAN Fakes")
print("  Dataset C = CIFAKE (Stable Diffusion images)")
print("="*60)
 
diff_res = {}
if not os.path.exists(cfg.DATASET_C):
    print("  ⚠️  Dataset C not found. Skipping.")
    with open(f'{PROJECT}/diffusion_results.json','w') as f: f.write('{}')
else:
    diff_ds  = DeepFakeDataset(cfg.DATASET_C, augment=False)
    diff_ldr = DataLoader(diff_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
 
    for name, binary in [('HSFAN',True),('CNN',False),('ResNet',False),('GAN',True)]:
        model = trained_models[name.lower()]
        model.eval()
        P, L, PR = [], [], []
        with torch.no_grad():
            for imgs, labels in diff_ldr:
                imgs = imgs.to(cfg.DEVICE); out = model(imgs)
                if binary:
                    probs = torch.sigmoid(out).squeeze(1).cpu().numpy()
                    preds = (probs > 0.5).astype(int)
                else:
                    p2 = torch.softmax(out,1).cpu().numpy()
                    probs = p2[:,1]; preds = p2.argmax(1)
                P.extend(preds.tolist()); L.extend(labels.int().numpy().tolist())
                PR.extend(probs.tolist())
 
        acc  = accuracy_score(L, P)
        f1   = f1_score(L, P, zero_division=0)
        prec = precision_score(L, P, zero_division=0)
        rec  = recall_score(L, P, zero_division=0)
        try:    auc = roc_auc_score(L, PR)
        except: auc = 0.0
 
        diff_res[name] = {'accuracy':round(acc,4), 'precision':round(prec,4),
                          'recall':round(rec,4), 'f1_score':round(f1,4),
                          'auc':round(auc,4),
                          'confusion_matrix':confusion_matrix(L,P).tolist(),
                          'dataset':'CIFAKE (Stable Diffusion)'}
        print(f"  {name}: Acc={acc:.4f} F1={f1:.4f} AUC={auc:.4f}")
 
    with open(f'{PROJECT}/diffusion_results.json','w') as f:
        json.dump(diff_res, f, indent=2)
    print(f"\n✅ Saved → diffusion_results.json")

  RESEARCH TEST 3: Diffusion Fakes vs GAN Fakes
  Dataset C = CIFAKE (Stable Diffusion images)
  [Dataset] dataset_C: 10000 real + 10000 fake = 20000 total
  HSFAN: Acc=0.9451 F1=0.9463 AUC=0.9887
  CNN: Acc=0.8679 F1=0.8757 AUC=0.9480
  ResNet: Acc=0.9859 F1=0.9860 AUC=0.9976
  GAN: Acc=0.8940 F1=0.8918 AUC=0.9596

✅ Saved → diffusion_results.json


In [13]:
## CELL 13 — Save Everything PERMANENTLY to Kaggle Output
##
## ✅ Saves all 4 model weights (.pth)
## ✅ Saves all 3 result JSON files
## ✅ Files go to /kaggle/working — Kaggle persists these
##    when you click "Save Version"
##
## HOW PERMANENT SAVING WORKS ON KAGGLE:
## 1. Run all cells (training + tests + this cell)
## 2. Click "Save Version" (top right) → "Save & Run All"
## 3. After it completes, go to Output tab of your notebook
## 4. Click ⋮ on the output dataset → "Add to notebook as input"
## 5. Next time: run ONLY Cells 1-7, then Cell 13B to load weights
##    and SKIP Cells 8-12 (no retraining needed!)
## ================================================================
import os, shutil
 
PROJECT  = '/kaggle/working/deepfake'
SAVE_DIR = '/kaggle/working'
 
print("💾 Saving all trained model weights and results...\n")
 
# ── Save all 4 model .pth files ──────────────────────────────────
for model_name in ['hsfan', 'cnn', 'resnet', 'gan']:
    # Try both locations (outputs/ folder and project root)
    for src in [f'{PROJECT}/outputs/{model_name}_model.pth',
                f'{PROJECT}/{model_name}_model.pth']:
        if os.path.exists(src):
            dest = f'{SAVE_DIR}/{model_name}_model.pth'
            shutil.copy(src, dest)
            size = os.path.getsize(dest) // 1024
            print(f"  ✅ {model_name}_model.pth saved  ({size} KB)")
            break
    else:
        print(f"  ⚠️  {model_name}_model.pth NOT FOUND — train first!")
 
print()
 
# ── Save all 3 result JSON files ─────────────────────────────────
for json_file in ['results.json', 'compression_results.json', 'diffusion_results.json']:
    src  = f'{PROJECT}/{json_file}'
    dest = f'{SAVE_DIR}/{json_file}'
    if os.path.exists(src):
        shutil.copy(src, dest)
        size = os.path.getsize(dest)
        print(f"  ✅ {json_file} saved  ({size} bytes)")
    else:
        print(f"  ⚠️  {json_file} NOT FOUND — run Tests 1, 2, 3 first!")
 
print()
print("="*60)
print("  ✅ ALL FILES SAVED TO /kaggle/working")
print("="*60)
print()
print("📌 NEXT STEP TO MAKE WEIGHTS PERMANENT:")
print("   1. Click 'Save Version' (top right) → Save & Run All")
print("   2. Wait for it to finish completely")
print("   3. Go to your notebook → Output tab (right panel)")
print("   4. Click ⋮ on the output → 'Add to notebook as input'")
print("   5. Note the path shown (e.g. /kaggle/input/deepfake-project/)")
print("   6. Update LOAD_DIR in Cell 13B with that path")
print("   7. Next session: run Cells 1-7, then Cell 13B, then 14-15")
print()
 
# ── Verify all files are present ─────────────────────────────────
print("🔍 Verifying saved files:\n")
all_ok = True
for fname in ['hsfan_model.pth', 'cnn_model.pth', 'resnet_model.pth', 'gan_model.pth',
              'results.json', 'compression_results.json', 'diffusion_results.json']:
    path = f'{SAVE_DIR}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  ✅ {fname:<35} ({size//1024} KB)")
    else:
        print(f"  ❌ MISSING: {fname}")
        all_ok = False
 
print()
if all_ok:
    print("🎉 ALL FILES VERIFIED — Safe to click Save Version now!")
else:
    print("⚠️  Some files are missing. Check the warnings above.")

💾 Saving all trained model weights and results...

  ✅ hsfan_model.pth saved  (21 KB)
  ✅ cnn_model.pth saved  (24 KB)
  ✅ resnet_model.pth saved  (7 KB)
  ✅ gan_model.pth saved  (3 KB)

  ✅ results.json saved  (998 bytes)
  ✅ compression_results.json saved  (624 bytes)
  ✅ diffusion_results.json saved  (1170 bytes)

  ✅ ALL FILES SAVED TO /kaggle/working

📌 NEXT STEP TO MAKE WEIGHTS PERMANENT:
   1. Click 'Save Version' (top right) → Save & Run All
   2. Wait for it to finish completely
   3. Go to your notebook → Output tab (right panel)
   4. Click ⋮ on the output → 'Add to notebook as input'
   5. Note the path shown (e.g. /kaggle/input/deepfake-project/)
   6. Update LOAD_DIR in Cell 13B with that path
   7. Next session: run Cells 1-7, then Cell 13B, then 14-15

🔍 Verifying saved files:

  ✅ hsfan_model.pth                     (21 KB)
  ✅ cnn_model.pth                       (24 KB)
  ✅ resnet_model.pth                    (7 KB)
  ✅ gan_model.pth                       (3 KB)
  ✅ r

In [14]:
## CELL 13B — Load Pre-trained Weights (USE THIS ON 2nd+ RUN)
##
## HOW TO USE:
## 1. After running Save Version once, go to Output tab
## 2. Click ⋮ → "Add to notebook as input dataset"
## 3. Update LOAD_DIR below with the path shown
## 4. Run ONLY: Cells 1,2,3,4,5,6,7 → Cell 13B → Cells 14,15
## 5. SKIP Cells 8,9,10,11,12 — no retraining needed!
## ================================================================
 
LOAD_DIR = '/kaggle/input/deepfake-project'   # ← UPDATE THIS PATH
 
def load_pretrained_models(load_dir):
    import json, torch
    PROJECT = '/kaggle/working/deepfake'
    print(f"📂 Loading from: {load_dir}\n")
    os.makedirs(f'{PROJECT}/outputs', exist_ok=True)
 
    model_classes = {
        'hsfan':  (HSFAN,                       True),
        'cnn':    (CNNModel,                    False),
        'resnet': (lambda: ResNetModel(False),  False),
        'gan':    (GANDetector,                 True),
    }
    _trained_models = {}
 
    for model_name, (model_cls, binary) in model_classes.items():
        src  = f'{load_dir}/{model_name}_model.pth'
        dest = f'{PROJECT}/{model_name}_model.pth'
        if not os.path.exists(src):
            print(f"  ❌ {model_name}_model.pth not found at {load_dir}")
            continue
        shutil.copy(src, dest)
        shutil.copy(src, f'{PROJECT}/outputs/{model_name}_model.pth')
        model = model_cls()
        model.load_state_dict(torch.load(dest, map_location=cfg.DEVICE))
        model.to(cfg.DEVICE).eval()
        _trained_models[model_name] = model
        size = os.path.getsize(dest) // 1024
        print(f"  ✅ {model_name}_model.pth loaded  ({size} KB)")
 
    print()
    for json_file in ['results.json', 'compression_results.json', 'diffusion_results.json']:
        src  = f'{load_dir}/{json_file}'
        dest = f'{PROJECT}/{json_file}'
        if os.path.exists(src):
            shutil.copy(src, dest)
            print(f"  ✅ {json_file} loaded")
        else:
            print(f"  ⚠️  {json_file} not found — using empty {{}}")
            with open(dest, 'w') as f: f.write('{}')
 
    print()
    print("="*60)
    print(f"  ✅ LOADED {len(_trained_models)}/4 models successfully")
    print("="*60)
    print("✅ Ready! Now run Cell 14 and Cell 15 directly.")
    return _trained_models
 
# ── UNCOMMENT THE LINE BELOW ON YOUR 2nd+ RUN ────────────────────
# trained_models = load_pretrained_models(LOAD_DIR)
# ─────────────────────────────────────────────────────────────────
print("✅ Cell 13B loader defined. Uncomment last line on 2nd+ run.")

✅ Cell 13B loader defined. Uncomment last line on 2nd+ run.


In [15]:
import os, time

os.system("pkill -9 -f app.py 2>/dev/null")
os.system("pkill -9 -f flask 2>/dev/null")
os.system("fuser -k -9 5000/tcp 2>/dev/null")
os.system("kill -9 $(lsof -t -i:5000) 2>/dev/null")
time.sleep(3)

# Verify port is free
result = os.popen("lsof -i:5000").read()
if result.strip():
    print("⚠️ Port still in use:", result)
else:
    print("✅ Port 5000 is now free")

sh: 1: lsof: not found


✅ Port 5000 is now free


/bin/sh: 1: lsof: not found


In [16]:
## ================================================================
## REPLACE ONLY THE BOTTOM SECTION of your Cell 14
## (from the line "# ── Load custom index.html" to the end)
## ================================================================

# ── Load custom index.html ─────────────────────────────────────
import os
PROJECT = '/kaggle/working/deepfake'

# ✅ FIX: Make sure templates folder exists before writing
os.makedirs(f'{PROJECT}/templates', exist_ok=True)

html_template = None
found_path    = None

# Known paths — your exact path is now first
known_paths = [
    '/kaggle/input/datasets/chethan200321/deepfake-html/index.html',  # ← YOUR EXACT PATH
    '/kaggle/input/deepfake-html/index.html',
    '/kaggle/input/deepfake-html/deepfake-html/index.html',
    '/kaggle/input/deepfakehtml/index.html',
    '/kaggle/input/deepfake-datasets/index.html',
]

print("🔍 Searching for index.html...")
for path in known_paths:
    if os.path.exists(path):
        found_path = path
        print(f"  ✅ Found: {path}")
        break
    else:
        print(f"  ✗ {path}")

# Deep search fallback
if not found_path:
    print("  → Deep search...")
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in files:
            if fname == 'index.html':
                found_path = os.path.join(root, fname)
                print(f"  ✅ Found: {found_path}")
                break
        if found_path:
            break

# Write to templates/
if found_path:
    with open(found_path) as f:
        html_template = f.read()
    with open(f'{PROJECT}/templates/index.html', 'w') as f:
        f.write(html_template)
    size = os.path.getsize(f'{PROJECT}/templates/index.html')
    print(f"  ✅ Written to templates/index.html  ({size} bytes)")
else:
    print("  ❌ index.html not found — using fallback")
    with open(f'{PROJECT}/templates/index.html', 'w') as f:
        f.write("""<!DOCTYPE html>
<html><head><meta charset="UTF-8"><title>DeepFake Detector</title></head>
<body><h1>DeepFake Detector</h1>
<form method="POST" enctype="multipart/form-data">
<input type="file" name="image" accept="image/*">
<button type="submit">Analyse</button></form>
{% if result %}
<h2>{{ result.prediction }} ({{ result.confidence }}%)</h2>
<p>Real: {{ result.real_prob }}% | Fake: {{ result.fake_prob }}%</p>
{% if result.heatmap_path %}<img src="{{ result.heatmap_path }}" width="256">{% endif %}
{% if result.lime_path %}<img src="{{ result.lime_path }}" width="256">{% endif %}
{% endif %}</body></html>""")

print("\n✅ All helper files written")

🔍 Searching for index.html...
  ✅ Found: /kaggle/input/datasets/chethan200321/deepfake-html/index.html
  ✅ Written to templates/index.html  (40318 bytes)

✅ All helper files written


In [17]:
## CELL 14 — Write HTML Template + All Helper Files  (FIXED)
## ================================================================
import os, shutil
 
PROJECT = '/kaggle/working/deepfake'
os.makedirs(f'{PROJECT}/model',            exist_ok=True)
os.makedirs(f'{PROJECT}/templates',        exist_ok=True)  # ✅ FIX: create before writing
os.makedirs(f'{PROJECT}/static/uploads',   exist_ok=True)
os.makedirs(f'{PROJECT}/static/heatmaps',  exist_ok=True)
os.makedirs(f'{PROJECT}/static/lime',      exist_ok=True)
 
with open(f'{PROJECT}/model/__init__.py','w') as f:
    f.write('from .hsfan import HSFAN\nfrom .cnn_model import CNNModel\n'
            'from .resnet_model import ResNetModel\nfrom .gan_detector import GANDetector\n')
 
with open(f'{PROJECT}/model/hsfan.py','w') as f:
    f.write('''import torch, torch.nn as nn, torch.nn.functional as F, timm
class AttentionBlock(nn.Module):
    def __init__(self,n):
        super().__init__()
        self.att=nn.Sequential(nn.Linear(n,n//4),nn.ReLU(),nn.Linear(n//4,n),nn.Sigmoid())
    def forward(self,x): return x*self.att(x)
class HSFAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial=timm.create_model("mobilenetv2_100",pretrained=False,num_classes=0)
        sd=self.spatial.num_features
        self.freq_branch=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.AdaptiveAvgPool2d((1,1)))
        c=sd+32
        self.attention=AttentionBlock(c)
        self.classifier=nn.Sequential(nn.Linear(c,256),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,1))
    def forward(self,x):
        sp=self.spatial(x)
        fft_mag=torch.fft.rfft2(x).abs().float()
        fft_mag=F.interpolate(fft_mag,size=(x.shape[2],x.shape[3]),mode="bilinear",align_corners=False)
        fr=self.freq_branch(fft_mag).view(x.size(0),-1)
        return self.classifier(self.attention(torch.cat([sp,fr],dim=1)))
''')
 
with open(f'{PROJECT}/model/cnn_model.py','w') as f:
    f.write('''import torch.nn as nn, timm
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=timm.create_model("efficientnet_b0",pretrained=False,num_classes=0)
        n=self.backbone.num_features
        self.classifier=nn.Sequential(
            nn.Linear(n,512),nn.BatchNorm1d(512),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(512,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self,x): return self.classifier(self.backbone(x))
''')
 
with open(f'{PROJECT}/model/resnet_model.py','w') as f:
    f.write('''import torch.nn as nn
from torchvision import models
class ResNetModel(nn.Module):
    def __init__(self,pretrained=False):
        super().__init__()
        w=models.ResNet18_Weights.DEFAULT if pretrained else None
        self.model=models.resnet18(weights=w)
        for n,p in self.model.named_parameters():
            if "layer4" not in n and "layer3" not in n and "fc" not in n: p.requires_grad=False
        self.model.fc=nn.Sequential(nn.Linear(self.model.fc.in_features,256),
            nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,2))
    def forward(self,x): return self.model(x)
''')
 
with open(f'{PROJECT}/model/gan_detector.py','w') as f:
    f.write('''import torch, torch.nn as nn
class SpectralBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d((4,4)))
        self.out=128*4*4
    def forward(self,x):
        return self.net(torch.log1p(torch.abs(torch.fft.fftshift(torch.fft.fft2(x))))).view(x.size(0),-1)
class CheckerboardBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,16,2,stride=2),nn.ReLU(),nn.Conv2d(16,32,2,stride=2),nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
        self.out=64*2*2
    def forward(self,x): return self.net(x).view(x.size(0),-1)
class SpatialBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
        self.out=64*2*2
    def forward(self,x): return self.net(x).view(x.size(0),-1)
class GANDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.spectral=SpectralBranch(); self.checker=CheckerboardBranch(); self.spatial=SpatialBranch()
        t=self.spectral.out+self.checker.out+self.spatial.out
        self.attention=nn.Sequential(nn.Linear(t,t//4),nn.ReLU(),nn.Linear(t//4,t),nn.Sigmoid())
        self.classifier=nn.Sequential(nn.Linear(t,512),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(512,128),nn.ReLU(),nn.Dropout(0.2),nn.Linear(128,1))
    def forward(self,x):
        f=torch.cat([self.spectral(x),self.checker(x),self.spatial(x)],dim=1)
        return self.classifier(f*self.attention(f))
''')
 
with open(f'{PROJECT}/gradcam.py','w') as f:
    f.write('''import torch, cv2, numpy as np, os
class GradCAM:
    def __init__(self,model,layer):
        self.model=model; self.g=self.a=None
        layer.register_forward_hook(lambda m,i,o: setattr(self,"a",o.detach()))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,"g",go[0].detach()))
    def generate(self,t):
        self.model.eval(); t=t.clone().requires_grad_(True)
        out=self.model(t)
        score=out[0][0] if out.shape[-1]==1 else out[0][out.argmax(1).item()]
        self.model.zero_grad(); score.backward()
        if self.g is None or self.a is None: return None
        w=self.g.cpu().numpy()[0].mean(axis=(1,2)); act=self.a.cpu().numpy()[0]
        cam=np.maximum(sum(w[i]*act[i] for i in range(len(w))),0)
        return cam/cam.max() if cam.max()>0 else cam
def get_gradcam_layer(model,t):
    t=t.lower()
    if t=="hsfan":  return model.freq_branch[4]
    if t=="resnet": return model.model.layer4[-1]
    if t=="cnn":    return model.backbone.conv_head
    if t=="gan":    return model.spectral.net[6]
def generate_heatmap(model,img_tensor,img_path,model_type,save_dir):
    os.makedirs(save_dir,exist_ok=True)
    try: layer=get_gradcam_layer(model,model_type)
    except Exception as e:
        print(f"[gradcam layer error] {e}"); return None
    cam=GradCAM(model,layer).generate(img_tensor)
    if cam is None: return None
    orig=cv2.imread(img_path)
    if orig is None: return None
    orig_disp=cv2.resize(orig,(256,256))
    h,w=orig_disp.shape[:2]
    heat=cv2.applyColorMap(np.uint8(255*cv2.resize(cam,(w,h))),cv2.COLORMAP_JET)
    rgb_orig=cv2.cvtColor(orig_disp,cv2.COLOR_BGR2RGB)
    rgb_heat=cv2.cvtColor(heat,cv2.COLOR_BGR2RGB)
    over=(0.55*rgb_orig+0.45*rgb_heat).astype(np.uint8)
    base=os.path.splitext(os.path.basename(img_path))[0]
    out_path=os.path.join(save_dir,f"{base}_gradcam.jpg")
    cv2.imwrite(out_path,cv2.cvtColor(over,cv2.COLOR_RGB2BGR))
    return out_path
''')
 
with open(f'{PROJECT}/lime_explainer.py','w') as f:
    f.write('''import numpy as np, cv2, os, torch
from torchvision import transforms
try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_OK=True
except: LIME_OK=False
IMG_SZ=128; DEVICE="cuda" if torch.cuda.is_available() else "cpu"
def generate_lime(model,img_path,model_type,binary=True,save_dir="static/lime",num_samples=200):
    if not LIME_OK: return None
    os.makedirs(save_dir,exist_ok=True)
    img=cv2.imread(img_path)
    if img is None: return None
    rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    tfm=transforms.Compose([transforms.ToPILImage(),transforms.Resize((IMG_SZ,IMG_SZ)),
        transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
    def predict_fn(images):
        batch=torch.stack([tfm(i) for i in images]).to(DEVICE)
        with torch.no_grad():
            out=model(batch)
            if binary:
                fp=torch.sigmoid(out).squeeze(1).cpu().numpy()
                return np.column_stack([1-fp,fp])
            return torch.softmax(out,dim=1).cpu().numpy()
    exp=lime_image.LimeImageExplainer()
    explanation=exp.explain_instance(rgb,predict_fn,top_labels=2,hide_color=0,num_samples=num_samples)
    temp,mask=explanation.get_image_and_mask(label=1,positive_only=True,num_features=8,hide_rest=False)
    lime_img=(mark_boundaries(temp/255.0,mask)*255).astype(np.uint8)
    lime_img=cv2.resize(lime_img,img.shape[1::-1])
    base=os.path.splitext(os.path.basename(img_path))[0]
    out_path=os.path.join(save_dir,f"{base}_lime.jpg")
    cv2.imwrite(out_path,cv2.cvtColor(lime_img,cv2.COLOR_RGB2BGR))
    return out_path
''')
 
# ── Load index.html — your exact path first, then deep search ──────
print("🔍 Loading index.html...")
HTML_PATH = '/kaggle/input/datasets/chethan200321/deepfake-html/index.html'
found_path = None
 
if os.path.exists(HTML_PATH):
    found_path = HTML_PATH
    print(f"  ✅ Found: {HTML_PATH}")
else:
    # Deep search fallback
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in files:
            if fname == 'index.html':
                found_path = os.path.join(root, fname)
                print(f"  ✅ Found via deep search: {found_path}")
                break
        if found_path:
            break
 
if found_path:
    with open(found_path) as f:
        content = f.read()
    with open(f'{PROJECT}/templates/index.html', 'w') as f:
        f.write(content)
    print(f"  ✅ Written → templates/index.html ({os.path.getsize(f'{PROJECT}/templates/index.html')} bytes)")
else:
    print("  ❌ index.html not found — using fallback")
    with open(f'{PROJECT}/templates/index.html', 'w') as f:
        f.write("""<!DOCTYPE html><html><head><meta charset="UTF-8">
<title>DeepFake Detector</title></head><body>
<form method="POST" enctype="multipart/form-data">
<input type="file" name="image"><button type="submit">Analyse</button></form>
{% if result %}<h2>{{result.prediction}} ({{result.confidence}}%)</h2>
{% if result.heatmap_path %}<img src="{{result.heatmap_path}}" width="256">{% endif %}
{% endif %}</body></html>""")
 
print("\n✅ All helper files written")


🔍 Loading index.html...
  ✅ Found: /kaggle/input/datasets/chethan200321/deepfake-html/index.html
  ✅ Written → templates/index.html (40318 bytes)

✅ All helper files written


In [18]:
## CELL 15 — Launch Flask App + Red/Black UI + Vivid Grad-CAM
## ✅ TPU first → GPU T4x2 fallback
## ✅ Vivid JET heatmap (matches reference Image 2)
## ✅ Improved ensemble: lower threshold for AI-smooth images
## ✅ Red and black web application (full rebuild)
## ================================================================
import sys, subprocess, time, os

PROJECT     = '/kaggle/working/deepfake'
NGROK_TOKEN = "3BTeGp52wk0v6mL6aOBfaRULicF_6JQHjYtw8dn5XbG39B4vT"

for folder in [PROJECT,
               f'{PROJECT}/static/uploads',
               f'{PROJECT}/static/heatmaps',
               f'{PROJECT}/static/lime',
               f'{PROJECT}/templates']:
    os.makedirs(folder, exist_ok=True)

os.system("pkill -9 -f app.py 2>/dev/null")
os.system("fuser -k -9 5000/tcp 2>/dev/null")
time.sleep(3)
print("✅ Port cleared")

os.system(f"ngrok authtoken {NGROK_TOKEN}")

# ── model files ───────────────────────────────────────────────────
os.makedirs(f'{PROJECT}/model', exist_ok=True)

with open(f'{PROJECT}/model/__init__.py','w') as _f:
    _f.write('from .hsfan import HSFAN\nfrom .cnn_model import CNNModel\n'
             'from .resnet_model import ResNetModel\nfrom .gan_detector import GANDetector\n')

HSFAN_PY = """import torch, torch.nn as nn, torch.nn.functional as F, timm
class AttentionBlock(nn.Module):
    def __init__(self,n):
        super().__init__()
        self.att=nn.Sequential(nn.Linear(n,n//4),nn.ReLU(),nn.Linear(n//4,n),nn.Sigmoid())
    def forward(self,x): return x*self.att(x)
class HSFAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial=timm.create_model("mobilenetv2_100",pretrained=False,num_classes=0)
        sd=self.spatial.num_features
        self.freq_branch=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.AdaptiveAvgPool2d((1,1)))
        c=sd+32
        self.attention=AttentionBlock(c)
        self.classifier=nn.Sequential(nn.Linear(c,256),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,1))
    def forward(self,x):
        sp=self.spatial(x)
        fft_mag=torch.fft.rfft2(x).abs().float()
        fft_mag=F.interpolate(fft_mag,size=(x.shape[2],x.shape[3]),mode="bilinear",align_corners=False)
        fr=self.freq_branch(fft_mag).view(x.size(0),-1)
        return self.classifier(self.attention(torch.cat([sp,fr],dim=1)))
"""

CNN_PY = """import torch.nn as nn, timm
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=timm.create_model("efficientnet_b0",pretrained=False,num_classes=0)
        n=self.backbone.num_features
        self.classifier=nn.Sequential(
            nn.Linear(n,512),nn.BatchNorm1d(512),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(512,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self,x): return self.classifier(self.backbone(x))
"""

RESNET_PY = """import torch.nn as nn
from torchvision import models
class ResNetModel(nn.Module):
    def __init__(self,pretrained=False):
        super().__init__()
        w=models.ResNet18_Weights.DEFAULT if pretrained else None
        self.model=models.resnet18(weights=w)
        for p in self.model.parameters():
            p.requires_grad=True
        self.model.fc=nn.Sequential(
            nn.Linear(self.model.fc.in_features,256),
            nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(256,64),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(64,2))
    def forward(self,x): return self.model(x)
"""

GAN_PY = """import torch, torch.nn as nn, torch.nn.functional as F
class SpectralBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.AdaptiveAvgPool2d((4,4)))
        self.out=128*4*4
    def forward(self,x):
        fft_mag=torch.fft.rfft2(x).abs().float()
        fft_mag=F.interpolate(fft_mag,size=(x.shape[2],x.shape[3]),mode="bilinear",align_corners=False)
        return self.net(torch.log1p(fft_mag)).view(x.size(0),-1)
class CheckerboardBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,16,2,stride=2),nn.ReLU(),nn.Conv2d(16,32,2,stride=2),nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
        self.out=64*2*2
    def forward(self,x): return self.net(x).view(x.size(0),-1)
class SpatialBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d((2,2)))
        self.out=64*2*2
    def forward(self,x): return self.net(x).view(x.size(0),-1)
class GANDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.spectral=SpectralBranch(); self.checker=CheckerboardBranch(); self.spatial=SpatialBranch()
        t=self.spectral.out+self.checker.out+self.spatial.out
        self.attention=nn.Sequential(nn.Linear(t,t//4),nn.ReLU(),nn.Linear(t//4,t),nn.Sigmoid())
        self.classifier=nn.Sequential(nn.Linear(t,512),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(512,128),nn.ReLU(),nn.Dropout(0.2),nn.Linear(128,1))
    def forward(self,x):
        f=torch.cat([self.spectral(x),self.checker(x),self.spatial(x)],dim=1)
        return self.classifier(f*self.attention(f))
"""

GRADCAM_PY = """import torch, cv2, numpy as np, os
class GradCAM:
    def __init__(self,model,layer):
        self.model=model; self.g=self.a=None
        layer.register_forward_hook(lambda m,i,o: setattr(self,"a",o.detach()))
        layer.register_full_backward_hook(lambda m,gi,go: setattr(self,"g",go[0].detach()))
    def generate(self,t):
        self.model.eval(); t=t.clone().requires_grad_(True)
        out=self.model(t)
        score=out[0][0] if out.shape[-1]==1 else out[0][out.argmax(1).item()]
        self.model.zero_grad(); score.backward()
        if self.g is None or self.a is None: return None
        w=self.g.cpu().numpy()[0].mean(axis=(1,2)); act=self.a.cpu().numpy()[0]
        cam=np.maximum(sum(w[i]*act[i] for i in range(len(w))),0)
        if cam.max()>0:
            cam=cam/cam.max()
            cam=np.power(cam,0.5)
        return cam
def get_gradcam_layer(model,t):
    t=t.lower()
    if t=="hsfan":  return model.freq_branch[4]
    if t=="resnet": return model.model.layer4[-1]
    if t=="cnn":    return model.backbone.conv_head
    if t=="gan":    return model.spectral.net[6]
def generate_heatmap(model,img_tensor,img_path,model_type,save_dir):
    os.makedirs(save_dir,exist_ok=True)
    try: layer=get_gradcam_layer(model,model_type)
    except Exception as e: print(f"[gradcam layer error] {e}"); return None
    cam=GradCAM(model,layer).generate(img_tensor)
    if cam is None: return None
    orig=cv2.imread(img_path)
    if orig is None: return None
    h,w=orig.shape[:2]
    cam_uint8=np.uint8(255*cv2.resize(cam,(w,h)))
    heat_bgr=cv2.applyColorMap(cam_uint8,cv2.COLORMAP_JET)
    orig_rgb=cv2.cvtColor(orig,cv2.COLOR_BGR2RGB)
    heat_rgb=cv2.cvtColor(heat_bgr,cv2.COLOR_BGR2RGB)
    overlay=np.clip(0.45*orig_rgb+0.55*heat_rgb,0,255).astype(np.uint8)
    base=os.path.splitext(os.path.basename(img_path))[0]
    out_path=os.path.join(save_dir,f"{base}_gradcam.jpg")
    cv2.imwrite(out_path,cv2.cvtColor(overlay,cv2.COLOR_RGB2BGR))
    return out_path
"""

LIME_PY = """import numpy as np, cv2, os, torch
from torchvision import transforms
try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_OK=True
except: LIME_OK=False
IMG_SZ=128; DEVICE="cuda" if torch.cuda.is_available() else "cpu"
def generate_lime(model,img_path,model_type,binary=True,save_dir="static/lime",num_samples=200):
    if not LIME_OK: return None
    os.makedirs(save_dir,exist_ok=True)
    img=cv2.imread(img_path)
    if img is None: return None
    rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    tfm=transforms.Compose([transforms.ToPILImage(),transforms.Resize((IMG_SZ,IMG_SZ)),
        transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
    def predict_fn(images):
        batch=torch.stack([tfm(i) for i in images]).to(DEVICE)
        with torch.no_grad():
            out=model(batch)
            if binary:
                fp=torch.sigmoid(out).squeeze(1).cpu().numpy()
                return np.column_stack([1-fp,fp])
            return torch.softmax(out,dim=1).cpu().numpy()
    exp=lime_image.LimeImageExplainer()
    explanation=exp.explain_instance(rgb,predict_fn,top_labels=2,hide_color=0,num_samples=num_samples)
    temp,mask=explanation.get_image_and_mask(label=1,positive_only=True,num_features=8,hide_rest=False)
    lime_img=(mark_boundaries(temp/255.0,mask)*255).astype(np.uint8)
    lime_img=cv2.resize(lime_img,img.shape[1::-1])
    base=os.path.splitext(os.path.basename(img_path))[0]
    out_path=os.path.join(save_dir,f"{base}_lime.jpg")
    cv2.imwrite(out_path,cv2.cvtColor(lime_img,cv2.COLOR_RGB2BGR))
    return out_path
"""

with open(f'{PROJECT}/model/hsfan.py','w') as _f: _f.write(HSFAN_PY)
with open(f'{PROJECT}/model/cnn_model.py','w') as _f: _f.write(CNN_PY)
with open(f'{PROJECT}/model/resnet_model.py','w') as _f: _f.write(RESNET_PY)
with open(f'{PROJECT}/model/gan_detector.py','w') as _f: _f.write(GAN_PY)
with open(f'{PROJECT}/gradcam.py','w') as _f: _f.write(GRADCAM_PY)
with open(f'{PROJECT}/lime_explainer.py','w') as _f: _f.write(LIME_PY)

# ── HTML template (red + black) ───────────────────────────────────
HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>DeepFake Detector — M-Tech</title>
<style>
*{margin:0;padding:0;box-sizing:border-box}
:root{--red:#e53935;--red2:#b71c1c;--red3:#ff5252;--bg:#0a0a0a;--card:#141414;--border:#2a0000;--text:#f0f0f0;--muted:#777}
body{background:var(--bg);color:var(--text);font-family:'Segoe UI',sans-serif;min-height:100vh}
header{background:linear-gradient(135deg,#0d0000,#1c0000);border-bottom:2px solid var(--red2);padding:14px 32px;display:flex;align-items:center;justify-content:space-between}
.logo{display:flex;align-items:center;gap:12px}
.logo-icon{width:42px;height:42px;background:var(--red2);border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:22px}
.logo-text h1{font-size:20px;font-weight:700;color:#fff;letter-spacing:1px}
.logo-text p{font-size:10px;color:var(--muted);letter-spacing:2px;text-transform:uppercase}
.hdr-right{display:flex;align-items:center;gap:18px}
.status{display:flex;align-items:center;gap:7px;font-size:12px;color:var(--red3)}
.dot{width:8px;height:8px;border-radius:50%;background:var(--red3);animation:pulse 1.5s infinite}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.35}}
.nav-tags{display:flex;gap:6px}
.nav-tags span{font-size:11px;color:var(--muted);padding:3px 10px;border:1px solid #2a0000;border-radius:20px}
.container{max-width:1100px;margin:0 auto;padding:30px 20px}

/* UPLOAD */
.upload-card{background:var(--card);border:2px dashed var(--red2);border-radius:16px;padding:44px;text-align:center;margin-bottom:28px}
.upload-card:hover{border-color:var(--red3)}
.upload-icon{font-size:52px;margin-bottom:14px;opacity:.7}
.upload-card h2{color:var(--red3);font-size:20px;margin-bottom:8px}
.upload-card p{color:var(--muted);font-size:13px;margin-bottom:22px}
#fileInput{display:none}
.file-label{background:var(--red2);color:#fff;padding:11px 30px;border-radius:8px;cursor:pointer;font-size:14px;font-weight:600;display:inline-block}
.file-label:hover{background:var(--red)}
#fileName{display:block;margin-top:10px;font-size:12px;color:var(--muted)}
.btn-analyse{background:linear-gradient(135deg,var(--red2),var(--red));color:#fff;border:none;padding:13px 40px;border-radius:8px;font-size:15px;font-weight:700;cursor:pointer;margin-top:18px;width:100%;max-width:340px;letter-spacing:1px}
.btn-analyse:hover{opacity:.85}

/* BANNER */
.result-banner{border-radius:12px;padding:22px 28px;margin-bottom:22px;display:flex;align-items:center;justify-content:space-between}
.banner-fake{background:linear-gradient(135deg,#1a0000,#2d0000);border:2px solid var(--red2)}
.banner-real{background:linear-gradient(135deg,#001a00,#003000);border:2px solid #2e7d32}
.verdict{display:flex;align-items:center;gap:16px}
.verdict-icon{font-size:38px;font-weight:900}
.verdict-label{font-size:40px;font-weight:900;letter-spacing:3px}
.verdict-label.fake{color:var(--red3)}
.verdict-label.real{color:#69f0ae}
.verdict-conf{font-size:13px;color:var(--muted);margin-top:4px}
.detected-by{text-align:right;font-size:11px;color:var(--muted);letter-spacing:1px}
.detected-by strong{display:block;color:var(--red3);font-size:11px;margin-top:5px}

/* STATS */
.stats-row{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-bottom:20px}
.stat-card{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:20px;text-align:center}
.stat-val{font-size:30px;font-weight:800;font-family:monospace}
.stat-val.fake{color:var(--red3)}
.stat-val.real{color:#69f0ae}
.stat-val.conf{color:#fff}
.stat-label{font-size:10px;color:var(--muted);letter-spacing:2px;margin-top:5px;text-transform:uppercase}

/* BARS */
.bars-card{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:22px;margin-bottom:20px}
.bar-row{display:flex;align-items:center;gap:12px;margin-bottom:14px}
.bar-label{width:50px;font-size:12px;font-weight:700;letter-spacing:1px}
.bar-label.real{color:#69f0ae}
.bar-label.fake{color:var(--red3)}
.bar-track{flex:1;height:14px;background:#1a1a1a;border-radius:7px;overflow:hidden}
.bar-fill{height:100%;border-radius:7px}
.bar-fill.real{background:linear-gradient(90deg,#1b5e20,#69f0ae)}
.bar-fill.fake{background:linear-gradient(90deg,var(--red2),var(--red3))}
.bar-pct{width:54px;text-align:right;font-size:14px;font-weight:700;font-family:monospace}

/* BREAKDOWN */
.breakdown{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:18px;margin-bottom:20px}
.breakdown h3{font-size:11px;color:var(--muted);letter-spacing:2px;text-transform:uppercase;margin-bottom:14px}
.model-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:10px}
.model-tile{background:#0e0e0e;border:1px solid #2a0000;border-radius:8px;padding:14px;text-align:center}
.model-tile .name{font-size:10px;color:var(--muted);letter-spacing:1px;text-transform:uppercase;margin-bottom:6px}
.model-tile .pct{font-size:20px;font-weight:800;color:var(--red3);font-family:monospace}

/* IMAGES */
.images-grid{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:20px}
.img-card{background:var(--card);border:1px solid var(--border);border-radius:10px;overflow:hidden}
.img-card-header{padding:10px 16px;background:#0d0000;font-size:10px;letter-spacing:2px;color:var(--muted);text-transform:uppercase;border-bottom:1px solid var(--border)}
.img-card img{width:100%;display:block}
.img-placeholder{height:220px;display:flex;align-items:center;justify-content:center;color:var(--muted);font-size:13px}
.upload-zone{background:var(--card);border:2px dashed var(--border);border-radius:14px;padding:32px;text-align:center}
.error-box{background:#1a0000;border:1px solid var(--red2);border-radius:8px;padding:14px 20px;margin-bottom:20px;color:var(--red3);font-size:14px}
footer{text-align:center;padding:24px;color:var(--muted);font-size:11px;border-top:1px solid #1a0000;margin-top:20px}
</style>
</head>
<body>
<header>
  <div class="logo">
    <div class="logo-icon">&#x1F50D;</div>
    <div class="logo-text">
      <h1>DeepFake Detector</h1>
      <p>M-Tech Research Project &bull; Computer Science</p>
    </div>
  </div>
  <div class="hdr-right">
    <div class="status"><div class="dot"></div> SYSTEM ACTIVE</div>
    <div class="nav-tags"><span>HSFAN</span><span>CNN</span><span>ResNet</span><span>GAN</span></div>
  </div>
</header>

<div class="container">
{% if error %}<div class="error-box">&#9888; {{ error }}</div>{% endif %}

{% if result and result.prediction %}
  {% set is_fake = result.prediction == 'FAKE' %}

  <div class="result-banner {{ 'banner-fake' if is_fake else 'banner-real' }}">
    <div class="verdict">
      <div class="verdict-icon">{{ '&#10007;' if is_fake else '&#10003;' }}</div>
      <div>
        <div class="verdict-label {{ 'fake' if is_fake else 'real' }}">{{ result.prediction }}</div>
        <div class="verdict-conf">{{ result.confidence }}% confidence</div>
      </div>
    </div>
    <div class="detected-by">DETECTED BY<strong>{{ result.model_type }}</strong></div>
  </div>

  <div class="stats-row">
    <div class="stat-card"><div class="stat-val fake">{{ result.fake_prob }}%</div><div class="stat-label">Fake Prob</div></div>
    <div class="stat-card"><div class="stat-val real">{{ result.real_prob }}%</div><div class="stat-label">Real Prob</div></div>
    <div class="stat-card"><div class="stat-val conf">{{ result.confidence }}%</div><div class="stat-label">Confidence</div></div>
  </div>

  <div class="bars-card">
    <div class="bar-row">
      <div class="bar-label real">REAL</div>
      <div class="bar-track"><div class="bar-fill real" style="width:{{ result.real_prob }}%"></div></div>
      <div class="bar-pct" style="color:#69f0ae">{{ result.real_prob }}%</div>
    </div>
    <div class="bar-row">
      <div class="bar-label fake">FAKE</div>
      <div class="bar-track"><div class="bar-fill fake" style="width:{{ result.fake_prob }}%"></div></div>
      <div class="bar-pct" style="color:var(--red3)">{{ result.fake_prob }}%</div>
    </div>
  </div>

  {% if result.model_breakdown %}
  <div class="breakdown">
    <h3>Model Breakdown — Fake Probability per Model</h3>
    <div class="model-grid">
      {% for name, pct in result.model_breakdown.items() %}
      <div class="model-tile"><div class="name">{{ name }}</div><div class="pct">{{ pct }}%</div></div>
      {% endfor %}
    </div>
  </div>
  {% endif %}

  <div class="images-grid">
    <div class="img-card">
      <div class="img-card-header">Original Image</div>
      {% if result.filename %}<img src="/static/uploads/{{ result.filename }}" alt="Original">
      {% else %}<div class="img-placeholder">No image</div>{% endif %}
    </div>
    <div class="img-card">
      <div class="img-card-header">Grad-CAM Heatmap</div>
      {% if result.heatmap_path %}<img src="{{ result.heatmap_path }}" alt="Grad-CAM">
      {% else %}<div class="img-placeholder">Heatmap not available</div>{% endif %}
    </div>
  </div>

  {% if result.lime_path %}
  <div class="img-card" style="margin-bottom:20px">
    <div class="img-card-header">LIME Superpixel Explanation</div>
    <img src="{{ result.lime_path }}" alt="LIME">
  </div>
  {% endif %}

  <div class="upload-zone">
    <form method="POST" enctype="multipart/form-data">
      <p style="color:var(--muted);margin-bottom:14px;font-size:13px">Analyse another image</p>
      <input type="file" name="image" id="fileInput2" accept="image/*" style="display:none"
             onchange="document.getElementById('fn2').textContent=this.files[0]?.name||''">
      <label for="fileInput2" class="file-label">Choose Image</label>
      <span id="fn2" style="display:block;margin-top:8px;font-size:12px;color:var(--muted)"></span>
      <br><button type="submit" class="btn-analyse">ANALYSE</button>
    </form>
  </div>

{% else %}
  <div class="upload-card">
    <div class="upload-icon">&#x1F4F7;</div>
    <h2>Upload Image for Analysis</h2>
    <p>Supports JPG, PNG, BMP, WEBP &bull; Max 16 MB</p>
    <form method="POST" enctype="multipart/form-data">
      <input type="file" name="image" id="fileInput" accept="image/*"
             onchange="document.getElementById('fileName').textContent=this.files[0]?.name||''">
      <label for="fileInput" class="file-label">Choose Image</label>
      <span id="fileName"></span>
      <br><button type="submit" class="btn-analyse">ANALYSE IMAGE</button>
    </form>
  </div>
{% endif %}
</div>
<footer>HSFAN Deepfake Detector &bull; M-Tech Final Year Project &bull; Computer Science 2026</footer>
</body>
</html>"""

with open(f'{PROJECT}/templates/index.html','w') as _f:
    _f.write(HTML)

# ── app.py ────────────────────────────────────────────────────────
APP_PY = f"""import os,sys,json,cv2,torch,numpy as np
sys.path.insert(0,'{PROJECT}')
os.chdir('{PROJECT}')
from flask import Flask,render_template,request
from torchvision import transforms
from model.hsfan import HSFAN
from model.cnn_model import CNNModel
from model.resnet_model import ResNetModel
from model.gan_detector import GANDetector
from gradcam import generate_heatmap
from lime_explainer import generate_lime

PROJECT='{PROJECT}'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
IMG_SZ=128
ALLOWED={{'jpg','jpeg','png','bmp','webp'}}

app=Flask(__name__,
    template_folder=PROJECT+'/templates',
    static_folder=PROJECT+'/static')
app.config['MAX_CONTENT_LENGTH']=16*1024*1024

def ok(fn): return '.' in fn and fn.rsplit('.',1)[1].lower() in ALLOWED

_cache={{}}
def load(mt):
    if mt in _cache: return _cache[mt]
    spec={{'hsfan':(HSFAN(),True),'cnn':(CNNModel(),False),
           'resnet':(ResNetModel(False),False),'gan':(GANDetector(),True)}}
    m,binary=spec[mt]
    path=PROJECT+'/'+mt+'_model.pth'
    if not os.path.exists(path): raise FileNotFoundError('Missing: '+path)
    m.load_state_dict(torch.load(path,map_location=DEVICE))
    m.to(DEVICE).eval()
    _cache[mt]=(m,binary)
    return m,binary

tfm=transforms.Compose([
    transforms.ToPILImage(),transforms.Resize((IMG_SZ,IMG_SZ)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])

def compute_frequency_score(img_bgr):
    gray=cv2.cvtColor(img_bgr,cv2.COLOR_BGR2GRAY).astype(np.float32)
    lap_var=cv2.Laplacian(gray,cv2.CV_32F).var()
    resized=np.float32(cv2.resize(gray,(64,64))/255.0)
    dct=cv2.dct(resized)
    total_e=np.sum(dct**2)+1e-8
    low_e=np.sum(dct[:8,:8]**2)
    dct_ratio=low_e/total_e
    kernel=np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]],dtype=np.float32)
    noise_std=float(cv2.filter2D(gray,-1,kernel).std())
    lap_norm=max(0.0,min(1.0,1.0-(lap_var/150.0)))
    noise_norm=max(0.0,min(1.0,1.0-(noise_std/20.0)))
    return float(0.4*lap_norm+0.3*dct_ratio+0.3*noise_norm)

def predict_ensemble(img_path):
    img=cv2.imread(img_path)
    if img is None: raise ValueError('Cannot read image')
    rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    t=tfm(rgb).unsqueeze(0).to(DEVICE)
    fake_probs={{}}
    for mt,binary in [('hsfan',True),('gan',True),('resnet',False),('cnn',False)]:
        try:
            m,_=load(mt)
            with torch.no_grad():
                out=m(t)
                fp=torch.sigmoid(out).item() if binary else torch.softmax(out,1)[0][1].item()
            fake_probs[mt]=fp
        except Exception as e:
            print(f'[warn] {{mt}} skipped: {{e}}')
            fake_probs[mt]=0.5
    freq_score=compute_frequency_score(img)
    ensemble_fake=(
        0.35*fake_probs.get('hsfan',0.5)+
        0.25*fake_probs.get('gan',0.5)+
        0.15*fake_probs.get('resnet',0.5)+
        0.05*fake_probs.get('cnn',0.5)+
        0.20*freq_score
    )
    if freq_score>0.55: threshold=0.32
    elif freq_score>0.40: threshold=0.38
    else: threshold=0.45
    pred='FAKE' if ensemble_fake>threshold else 'REAL'
    conf=round((ensemble_fake if pred=='FAKE' else 1-ensemble_fake)*100,2)
    cam,lime=None,None
    try:
        m_h,_=load('hsfan')
        t2=tfm(rgb).unsqueeze(0).to(DEVICE)
        ca=generate_heatmap(m_h,t2,img_path,'hsfan',PROJECT+'/static/heatmaps')
        li=generate_lime(m_h,img_path,'hsfan',True,PROJECT+'/static/lime',200)
        if ca and os.path.exists(ca): cam='/static/heatmaps/'+os.path.basename(ca)
        if li and os.path.exists(li): lime='/static/lime/'+os.path.basename(li)
    except Exception as xe: print(f'[xai error] {{xe}}')
    return {{'prediction':pred,
            'fake_prob':round(ensemble_fake*100,2),
            'real_prob':round((1-ensemble_fake)*100,2),
            'confidence':conf,
            'model_type':'ENSEMBLE (HSFAN+GAN+ResNet+CNN+Freq)',
            'heatmap_path':cam,'lime_path':lime,'filename':'',
            'freq_score':round(freq_score*100,1),
            'threshold_used':round(threshold*100,1),
            'model_breakdown':{{
                'HSFAN':round(fake_probs.get('hsfan',0.5)*100,1),
                'GAN':round(fake_probs.get('gan',0.5)*100,1),
                'ResNet':round(fake_probs.get('resnet',0.5)*100,1),
                'CNN':round(fake_probs.get('cnn',0.5)*100,1),
            }}}}

def load_json(fn):
    try:
        with open(PROJECT+'/'+fn) as _f: return json.load(_f)
    except: return {{}}

@app.route('/',methods=['GET','POST'])
def index():
    result,error={{}},None
    if request.method=='POST':
        file=request.files.get('image')
        if not file or not file.filename: error='No file selected'
        elif not ok(file.filename): error='Use JPG PNG BMP WEBP'
        else:
            safe=file.filename.replace(' ','_')
            path=PROJECT+'/static/uploads/'+safe
            file.save(path)
            try:
                result=predict_ensemble(path)
                result['filename']=safe
            except Exception as e: error=str(e)
    return render_template('index.html',result=result,error=error,
        comparison_data=load_json('results.json'),
        compression_data=load_json('compression_results.json'),
        diffusion_data=load_json('diffusion_results.json'))

if __name__=='__main__':
    print('[App] Flask ready on port 5000')
    app.run(debug=False,host='0.0.0.0',port=5000,use_reloader=False)
"""

with open(f'{PROJECT}/app.py','w') as _f:
    _f.write(APP_PY)

print("✅ All files written (models, gradcam, lime, HTML, app.py)")

# ── Start Flask ───────────────────────────────────────────────────
flask_proc = subprocess.Popen(
    [sys.executable, f'{PROJECT}/app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print("⏳ Waiting for Flask to start...")
import socket
for _ in range(30):
    time.sleep(1)
    try:
        with socket.create_connection(("127.0.0.1", 5000), timeout=1):
            print("✅ Flask is up on port 5000")
            break
    except:
        pass

try:
    from pyngrok import ngrok
    tunnel = ngrok.connect(5000)
    url = tunnel.public_url if hasattr(tunnel,'public_url') else str(tunnel)
    print("=" * 55)
    print(f"  🌐  PUBLIC URL: {url}")
    print("=" * 55)
except Exception as ne:
    print(f"⚠️  ngrok error: {ne}")
    print("   App running at http://localhost:5000")


✅ Port cleared
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml                                
✅ All files written (models, gradcam, lime, HTML, app.py)
⏳ Waiting for Flask to start...
✅ Flask is up on port 5000
  🌐  PUBLIC URL: https://rolf-invincible-superintensely.ngrok-free.dev
